In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [28]:
#load calculated prop metrics
df_all_entries = pd.read_csv("/n/groups/marks/projects/viral_families/priority-viruses/results/summary/neff_supp_viralDMS_granular.csv")
df_all_entries["Bitscore"] = df_all_entries["Bitscore"].str.replace("b", "", regex=False).astype(float)

In [29]:
#merge with EVCoupling Spearman Results
evh_spearman = pd.read_csv("/n/groups/marks/projects/viral_families/priority-viruses/results/summary/ali_selection_summary.csv")

In [30]:
df_all_entries = df_all_entries.merge(evh_spearman, how='left', left_on=['DMS ID', 'Database', 'Bitscore'], right_on=['DMS ID', 'Database', 'Bitscore'])

In [31]:
df_all_entries

,Unnamed: 0,DMS ID,Database,Bitscore,neff,neff99,neff95,neff90,neff85,neff80,...,neff30,EVCouplings Spearman,len_seq,Virus,assay,log_neff90,Num_Seq,Prop_90,Len_Seq,Neff90
0,0,LASSA_GP_Carr,uniref100,0.03,633.637818,1.000000,2.733766,160.693312,296.815407,307.815407,...,561.971151,0.329820,491.0,Lassa,fitness,0.010345,1082.0,0.346580,491.0,160.693312
1,1,LASSA_GP_Carr,uniref100,0.04,850.851085,1.085958,82.387978,270.998437,286.998437,295.998437,...,618.851085,0.316467,491.0,Lassa,fitness,0.011410,1373.0,0.405681,491.0,270.998437
2,2,LASSA_GP_Carr,uniref100,0.05,654.212232,1.085958,82.387978,270.998437,286.998437,295.998437,...,645.212232,0.314998,491.0,Lassa,fitness,0.011410,1136.0,0.490317,491.0,270.998437
3,3,LASSA_GP_Carr,uniref100,0.10,654.212232,1.085958,82.387978,270.998437,286.998437,295.998437,...,646.212232,0.336073,491.0,Lassa,fitness,0.011410,1136.0,0.490317,491.0,270.998437
4,4,LASSA_GP_Carr,uniref100,0.30,650.816351,1.085958,82.387978,270.998437,286.998437,295.998437,...,640.149684,0.320777,491.0,Lassa,fitness,0.011410,1136.0,0.490317,491.0,270.998437
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653,653,LAMBDA_HCP_Tsuboyama,uniref90,0.50,1014.000000,1.000000,1.000000,4.000000,11.000000,19.000000,...,894.000000,0.435424,55.0,Escherichia phage lambda,stability,0.025205,1025.0,0.006829,55.0,4.000000
654,654,LAMBDA_HCP_Tsuboyama,uniref_bfd_mgnify,0.50,4121.000000,1.000000,118.000000,142.000000,224.000000,280.000000,...,3642.000000,0.257290,55.0,Escherichia phage lambda,stability,0.090106,5457.0,0.045629,55.0,142.000000
655,655,BP434_RPC1_Tsuboyama,uniref100,0.50,625019.000000,1.000000,14.000000,21.000000,26.000000,36.000000,...,301765.000000,0.552712,61.0,Enterobacteria phage 434,stability,0.049910,918385.0,0.000063,61.0,21.000000
656,656,BP434_RPC1_Tsuboyama,uniref90,0.30,780890.000000,1.000000,3.000000,5.000000,6.000000,13.000000,...,597949.000000,0.585581,61.0,Enterobacteria phage 434,stability,0.026384,818078.0,0.000007,61.0,5.000000


In [32]:
# --- config ---
y_var = "EVCouplings Spearman"
neff_columns = [
    "neff99", "neff95", "neff90", "neff85", "neff80", "neff75",
    "neff70", "neff65", "neff60", "neff55", "neff50", "neff30"
]
DIFF_THRESH = 0.1  # significance = (per DMS ID & x_var) spearman range ≥ 0.05

# --- data prep ---
df = df_all_entries.copy()
for col in [y_var] + neff_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

dfs_by_x = {}       # x_var -> per-x table (per DMS)
summary_rows = []   # one row per Neff (x_var)

# --- per x_var loop ---
for x_var in neff_columns:
    rows = []
    for dms_id, df_i in df.groupby("DMS ID"):
        # finite subset for this DMS & x_var
        mask = np.isfinite(df_i[x_var]) & np.isfinite(df_i[y_var])
        sub = df_i.loc[mask, [x_var, y_var]]

        # per-(DMS ID, x_var) spearman range (based only on this subset)
        if len(sub):
            spearman_diff_id = float(sub[y_var].max() - sub[y_var].min())
        else:
            spearman_diff_id = np.nan

        # correlation (only if both axes vary and ≥2 points)
        n = len(sub)
        x_unique = sub[x_var].nunique(dropna=True)
        y_unique = sub[y_var].nunique(dropna=True)
        if n >= 2 and x_unique > 1 and y_unique > 1:
            rho, p = stats.spearmanr(sub[x_var], sub[y_var], nan_policy="omit")
        else:
            rho, p = np.nan, np.nan

        rows.append({
            "DMS ID": dms_id,
            "x_var": x_var,
            "spearman_rho": rho,
            "p_value": p,                # kept for reference; NOT used for significance
            "n_points": n,
            "x_unique": x_unique,
            "y_unique": y_unique,
            "x_min": sub[x_var].min() if n else np.nan,
            "x_max": sub[x_var].max() if n else np.nan,
            "y_min": sub[y_var].min() if n else np.nan,
            "y_max": sub[y_var].max() if n else np.nan,
            "spearman_diff_id": spearman_diff_id,  # per-(DMS ID, x_var) range
        })

    # table for this x_var (per-DMS rows)
    df_corr = pd.DataFrame(rows).sort_values(["DMS ID"]).reset_index(drop=True)

    # significance flag based on diff ≥ 0.05 AND having a finite rho
    df_corr["sig_diff"] = df_corr["spearman_diff_id"].ge(DIFF_THRESH) & df_corr["spearman_rho"].notna()

    # save per-x_var table
    out_path = f"spearman_{x_var}_vs_{y_var}.csv"
    df_corr.to_csv(out_path, index=False)
    print(f"Saved: {out_path}  (valid rhos: {(~df_corr['spearman_rho'].isna()).sum()}/{len(df_corr)})")

    dfs_by_x[x_var] = df_corr

    # ---- per-Neff summary (single row) ----
    valid = df_corr["spearman_rho"].notna()
    sig_mask = df_corr["sig_diff"] & valid
    r_sig = df_corr.loc[sig_mask, "spearman_rho"]

    summary_rows.append({
        "x_var": x_var,
        "n_rows": int(len(df_corr)),
        "n_valid_rho": int(valid.sum()),
        "n_sig_diff>=0.05": int(sig_mask.sum()),
        "n_pos_sig_diff>=0.05": int((r_sig > 0).sum()),
        "n_neg_sig_diff>=0.05": int((r_sig < 0).sum()),
        "n_zero_sig_diff>=0.05": int((r_sig == 0).sum()),
        "valid_sig_diff>=0.05": int(r_sig.shape[0]),
        # Optional summary stats:
        "mean_rho_all_valid": float(df_corr.loc[valid, "spearman_rho"].mean()) if valid.any() else np.nan,
        "median_rho_all_valid": float(df_corr.loc[valid, "spearman_rho"].median()) if valid.any() else np.nan,
    })

# -------- one-row-per-Neff summary table --------
summary_df = pd.DataFrame(summary_rows)
summary_df["x_var"] = pd.Categorical(summary_df["x_var"], categories=neff_columns, ordered=True)
summary_df = summary_df.sort_values("x_var").reset_index(drop=True)




Saved: spearman_neff99_vs_EVCouplings Spearman.csv  (valid rhos: 37/47)
Saved: spearman_neff95_vs_EVCouplings Spearman.csv  (valid rhos: 41/47)
Saved: spearman_neff90_vs_EVCouplings Spearman.csv  (valid rhos: 43/47)
Saved: spearman_neff85_vs_EVCouplings Spearman.csv  (valid rhos: 44/47)
Saved: spearman_neff80_vs_EVCouplings Spearman.csv  (valid rhos: 44/47)
Saved: spearman_neff75_vs_EVCouplings Spearman.csv  (valid rhos: 45/47)
Saved: spearman_neff70_vs_EVCouplings Spearman.csv  (valid rhos: 45/47)
Saved: spearman_neff65_vs_EVCouplings Spearman.csv  (valid rhos: 45/47)
Saved: spearman_neff60_vs_EVCouplings Spearman.csv  (valid rhos: 45/47)
Saved: spearman_neff55_vs_EVCouplings Spearman.csv  (valid rhos: 45/47)
Saved: spearman_neff50_vs_EVCouplings Spearman.csv  (valid rhos: 45/47)
Saved: spearman_neff30_vs_EVCouplings Spearman.csv  (valid rhos: 45/47)


In [33]:
#summarize with graphs
summary_df

,x_var,n_rows,n_valid_rho,n_sig_diff>=0.05,n_pos_sig_diff>=0.05,n_neg_sig_diff>=0.05,n_zero_sig_diff>=0.05,valid_sig_diff>=0.05,mean_rho_all_valid,median_rho_all_valid
0,neff99,47,37,27,7,20,0,27,-0.228490,-0.382301
1,neff95,47,41,29,21,8,0,29,0.122485,0.250721
2,neff90,47,43,29,18,11,0,29,0.076090,0.207430
3,neff85,47,44,30,19,11,0,30,0.141101,0.314517
4,neff80,47,44,30,19,11,0,30,0.092906,0.089242
5,neff75,47,45,31,18,13,0,31,0.042281,-0.100000
6,neff70,47,45,31,16,15,0,31,0.012720,-0.096025
7,neff65,47,45,31,14,16,1,31,-0.030674,-0.120879
8,neff60,47,45,31,13,18,0,31,-0.029778,-0.032967
9,neff55,47,45,31,13,18,0,31,-0.070183,-0.149639
